In [12]:
# Install dependencies as needed:
# pip install kagglehub[hf-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "AAPL.csv"

# Load the latest version
hf_dataset = kagglehub.load_dataset(
  KaggleDatasetAdapter.HUGGING_FACE,
  "caesarmario/apple-stock-historical-price",
  file_path,
  # Provide any additional arguments like
  # sql_query, hf_kwargs, or pandas_kwargs. See
  # the documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterhugging_face
)

print("Hugging Face Dataset:", hf_dataset)

/tmp/ipykernel_975/2548405245.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  hf_dataset = kagglehub.load_dataset(


Using Colab cache for faster access to the 'apple-stock-historical-price' dataset.
Hugging Face Dataset: Dataset({
    features: ['Date', 'open', 'high', 'low', 'close', 'adjclose', 'volume', 'ingested_at_utc'],
    num_rows: 2568
})


In [13]:
import pandas as pd

start_date_str = '2022-03-20 00:00:00 UTC'
end_date_str = '2023-03-19 23:59:59 UTC'

def filter_by_date(example):
    # Localize the example date to UTC, assuming it's timezone-naive
    date = pd.to_datetime(example['Date']).tz_localize('UTC')
    # Start and end dates are already timezone-aware due to 'UTC' in the string, so just parse them
    start_date = pd.to_datetime(start_date_str)
    end_date = pd.to_datetime(end_date_str)
    return start_date <= date <= end_date

APPL_price_data = hf_dataset.filter(filter_by_date)

print("Filtered Hugging Face Dataset:", APPL_price_data)
print("First 5 records of filtered data:", APPL_price_data[:5])

Filter:   0%|          | 0/2568 [00:00<?, ? examples/s]

Filtered Hugging Face Dataset: Dataset({
    features: ['Date', 'open', 'high', 'low', 'close', 'adjclose', 'volume', 'ingested_at_utc'],
    num_rows: 250
})
First 5 records of filtered data: {'Date': ['2022-03-21', '2022-03-22', '2022-03-23', '2022-03-24', '2022-03-25'], 'open': [163.50999450683594, 165.50999450683594, 167.99000549316406, 171.05999755859375, 173.8800048828125], 'high': [166.35000610351562, 169.4199981689453, 172.63999938964844, 174.13999938964844, 175.27999877929688], 'low': [163.00999450683594, 164.91000366210938, 167.64999389648438, 170.2100067138672, 172.75], 'close': [165.3800048828125, 168.82000732421875, 170.2100067138672, 174.07000732421875, 174.72000122070312], 'adjclose': [162.03677368164062, 165.4072265625, 166.76914978027344, 170.55108642578125, 171.1879425048828], 'volume': [95811400, 81532000, 98062700, 90131400, 80546200], 'ingested_at_utc': ['2026-03-22 06:29:28.833352+00:00', '2026-03-22 06:29:28.833352+00:00', '2026-03-22 06:29:28.833352+00:00', '202

In [14]:
import pandas as pd

daily_sentiment_df = pd.read_csv('/content/daily_sentiment.csv')
print("Daily Sentiment DataFrame head:")
print(daily_sentiment_df.head())

Daily Sentiment DataFrame head:
          day  avg_positive_score  avg_negative_score  avg_neutral_score
0  2022-06-03            0.042049            0.729936           0.228015
1  2022-06-04            0.060833            0.064788           0.874379
2  2022-06-05            0.543850            0.015138           0.441012
3  2022-06-06            0.330284            0.333007           0.336709
4  2022-06-07            0.218690            0.249324           0.531986


In [15]:
APPL_price_df = APPL_price_data.to_pandas()

# Convert 'Date' column to datetime in APPL_price_df
APPL_price_df['Date'] = pd.to_datetime(APPL_price_df['Date'])

# Convert 'day' column to datetime in daily_sentiment_df
daily_sentiment_df['day'] = pd.to_datetime(daily_sentiment_df['day'])

# Merge the two DataFrames on their date columns
# Changed to a 'left' merge to keep all dates from daily_sentiment_df
merged_df = pd.merge(daily_sentiment_df, APPL_price_df, left_on='day', right_on='Date', how='left')

print("Merged DataFrame head:")
print(merged_df.head())
print("Merged DataFrame info:")
merged_df.info()

Merged DataFrame head:
         day  avg_positive_score  avg_negative_score  avg_neutral_score  \
0 2022-06-03            0.042049            0.729936           0.228015   
1 2022-06-04            0.060833            0.064788           0.874379   
2 2022-06-05            0.543850            0.015138           0.441012   
3 2022-06-06            0.330284            0.333007           0.336709   
4 2022-06-07            0.218690            0.249324           0.531986   

        Date        open        high         low       close    adjclose  \
0 2022-06-03  146.899994  147.970001  144.460007  145.380005  142.650375   
1        NaT         NaN         NaN         NaN         NaN         NaN   
2        NaT         NaN         NaN         NaN         NaN         NaN   
3 2022-06-06  147.029999  148.570007  144.899994  146.139999  143.396072   
4 2022-06-07  144.350006  149.000000  144.100006  148.710007  145.917816   

       volume                   ingested_at_utc  
0  88570300.0  2026

In [16]:
import pandas as pd

# 1. Load your dataset
#df = pd.read_csv('sentiment_and_price_data.csv')

# 2. Convert the 'day' column to datetime and sort it
# This ensures that 'forward' actually means the next chronological day
merged_df['day'] = pd.to_datetime(merged_df['day'])
final_df = merged_df.sort_values('day')

# 3. Define the columns that have missing values on weekends
price_columns = ['open', 'high', 'low', 'close', 'adjclose', 'volume']

# 4. Apply the forward-fill method
# This fills NaN values with the value from the previous row
final_df[price_columns] = final_df[price_columns].ffill()

# 5. Save the cleaned data to a new file
#final_df.to_csv('sentiment_and_price_data_filled.csv', index=False)

print("Weekend price data has been filled successfully.")

Weekend price data has been filled successfully.


In [17]:
final_df = final_df.drop(columns=['Date'])

print("Merged DataFrame after dropping 'Date' column:")
print(final_df.head())#
print("\nMerged DataFrame info after dropping 'Date' column:")
final_df.info()

Merged DataFrame after dropping 'Date' column:
         day  avg_positive_score  avg_negative_score  avg_neutral_score  \
0 2022-06-03            0.042049            0.729936           0.228015   
1 2022-06-04            0.060833            0.064788           0.874379   
2 2022-06-05            0.543850            0.015138           0.441012   
3 2022-06-06            0.330284            0.333007           0.336709   
4 2022-06-07            0.218690            0.249324           0.531986   

         open        high         low       close    adjclose      volume  \
0  146.899994  147.970001  144.460007  145.380005  142.650375  88570300.0   
1  146.899994  147.970001  144.460007  145.380005  142.650375  88570300.0   
2  146.899994  147.970001  144.460007  145.380005  142.650375  88570300.0   
3  147.029999  148.570007  144.899994  146.139999  143.396072  71598400.0   
4  144.350006  149.000000  144.100006  148.710007  145.917816  67808200.0   

                    ingested_at_utc  
0

In [18]:
import pandas as pd

# --- Validation Phase ---

# 1. Check for missing business dates in the sequence of merged_df
min_date = final_df['day'].min()
max_date = final_df['day'].max()

# Create a complete range of business days between min and max date
# Stock data typically only has entries for business days
full_business_days_range = pd.date_range(start=min_date, end=max_date, freq='B')

# Get the unique business days present in merged_df
# We normalize to just the date part for comparison
present_business_days = pd.Series(final_df['day'].dt.normalize().unique())

# Find missing business days within the expected range
missing_business_days = full_business_days_range.difference(present_business_days)

if not missing_business_days.empty:
    print(f"Validation Warning: The following business days are missing in final_df: {missing_business_days.strftime('%Y-%m-%d').tolist()}")
    # Depending on requirements, you might want to raise an error here
    # raise ValueError(f"Missing business days found: {missing_business_days.strftime('%Y-%m-%d').tolist()}")
else:
    print("Validation: No missing business days found in the sequence of merged_df.")

# 2. Verify that every date in merged_df exists in daily_sentiment_df
# This condition is implicitly guaranteed by the 'inner' merge operation
# used to create merged_df. An 'inner' merge only keeps rows where the
# merge key (date) is present in both dataframes.
# So, if a date exists in merged_df, it must have existed in daily_sentiment_df.
merged_dates_set = set(final_df['day'].dt.normalize())
daily_sentiment_dates_set = set(daily_sentiment_df['day'].dt.normalize())

if not merged_dates_set.issubset(daily_sentiment_dates_set):
    # This block should ideally never be reached with an 'inner' merge.
    # If it is, there might be an issue with how merged_df was created or date types.
    print("Validation Error: Some dates in merged_df do not exist in daily_sentiment_df. This indicates an unexpected issue with the merge operation.")
    # raise ValueError("Date integrity check failed for merged_df against daily_sentiment_df.")
else:
    print("Validation: All dates in merged_df also exist in daily_sentiment_df (as expected from the inner merge).")

# If validation passes (or if warnings are acceptable), then save the file
print("\nValidation phase completed. Saving the file.")
final_df.to_csv('/content/sentiment_and_price_data.csv', index=False)
print('final_df.csv saved successfully to /content/')


Validation: No missing business days found in the sequence of merged_df.
Validation: All dates in merged_df also exist in daily_sentiment_df (as expected from the inner merge).

Validation phase completed. Saving the file.
final_df.csv saved successfully to /content/


In [19]:
print(f"Number of rows in APPL_price_df: {APPL_price_df.shape[0]}")
print(f"Number of rows in daily_sentiment_df: {daily_sentiment_df.shape[0]}")
print(f"Number of rows in merged_df: {merged_df.shape[0]}")

Number of rows in APPL_price_df: 250
Number of rows in daily_sentiment_df: 286
Number of rows in merged_df: 286
